In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Monte Carlo bias/variance experiment for longitudinal-modularity null expectation.

Input CSV must contain columns:
  - source
  - destination
  - timestamp

We randomly assign soft community membership vectors (Dirichlet) to each node
at each interaction time it appears (source and destination separately).
Then compute T_{u in C} by:
  T_{u,C} += p_{u,C}(t_i) * (t_{i+1} - t_i)  for consecutive events of node u.

Compute:
  E_exact = sum_C sum_{u,v} (k_u k_v / (2m)) * sqrt( (T_{u,C} T_{v,C}) / T )

We compute E_exact efficiently using factorization:
  E_exact = (1/(2m*sqrt(T))) * sum_C (sum_u k_u sqrt(T_{u,C}))^2

Monte Carlo estimator (degree-proportional node-pair sampling):
  p(u)=k_u/(2m), u,v iid ~ p
  E_hat = (2m) * (1/M) * sum_i sum_C sqrt(T_{u_i,C} T_{v_i,C} / T)

We repeat NUM_TRIALS for each M, report mean/var of error, and draw histograms.
"""

import os
import argparse
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _ensure_timestamp_numeric(ts: pd.Series) -> np.ndarray:
    """Return timestamps as float64 (seconds if datetime-like)."""
    if np.issubdtype(ts.dtype, np.number):
        arr = ts.to_numpy(dtype=np.float64)
        return arr

    # try parse datetime
    dt = pd.to_datetime(ts, errors="coerce", utc=True)
    if dt.isna().any():
        raise ValueError(
            "timestamp column is neither numeric nor fully parseable as datetime."
        )
    # seconds since epoch as float
    arr = (dt.view("int64") / 1e9).astype(np.float64)
    return arr


def compute_T_uc(
    df: pd.DataFrame,
    K: int,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray, float]:
    """
    Compute:
      - T_uc: shape (N, K)
      - deg:  shape (N,)
      - T_total: scalar, global time span
    Random soft assignments are generated per row for source and destination.
    """
    needed = {"source", "destination", "timestamp"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"CSV missing columns: {missing}")

    # sort by timestamp
    df = df.copy()
    df["__ts"] = _ensure_timestamp_numeric(df["timestamp"])
    df = df.sort_values("__ts").reset_index(drop=True)

    # nodes
    nodes = pd.unique(df[["source", "destination"]].to_numpy().ravel())
    node_to_idx = {u: i for i, u in enumerate(nodes)}
    N = len(nodes)

    # total time span
    t0 = float(df["__ts"].iloc[0])
    t1 = float(df["__ts"].iloc[-1])
    T_total = max(t1 - t0, 0.0)
    if T_total <= 0:
        raise ValueError("Total time span T_total <= 0. Need at least 2 distinct timestamps.")

    # random memberships per interaction endpoint
    rng = np.random.default_rng(seed)
    P_src = rng.dirichlet(alpha=np.ones(K), size=len(df))
    P_dst = rng.dirichlet(alpha=np.ones(K), size=len(df))

    # collect per-node events: list of (timestamp, prob_vector_at_that_time)
    events = defaultdict(list)
    for i, row in df.iterrows():
        ts = float(row["__ts"])
        s = row["source"]
        d = row["destination"]
        events[s].append((ts, P_src[i]))
        events[d].append((ts, P_dst[i]))

    # compute T_uc
    T_uc = np.zeros((N, K), dtype=np.float64)
    for u, evs in events.items():
        evs.sort(key=lambda x: x[0])
        if len(evs) < 2:
            continue
        idx = node_to_idx[u]
        for (t_i, p_i), (t_next, _) in zip(evs[:-1], evs[1:]):
            dt = float(t_next - t_i)
            if dt <= 0:
                continue
            # your definition: use p at time t_i multiplied by time gap
            T_uc[idx] += p_i * dt

    # degrees from interactions (undirected count: each row contributes 1 to source and 1 to destination)
    deg = np.zeros(N, dtype=np.float64)
    for _, row in df.iterrows():
        deg[node_to_idx[row["source"]]] += 1.0
        deg[node_to_idx[row["destination"]]] += 1.0

    return T_uc, deg, T_total


def exact_E(T_uc: np.ndarray, deg: np.ndarray, T_total: float) -> float:
    """
    Efficient exact computation:
      E = 1/(2m*sqrt(T)) * sum_C (sum_u k_u sqrt(T_{u,C}))^2
    where 2m = sum_u k_u
    """
    m2 = float(deg.sum())
    if m2 <= 0:
        return 0.0

    # guard: sqrt(0)=0 ok
    sqrt_Tuc = np.sqrt(np.maximum(T_uc, 0.0))
    S = (deg[:, None] * sqrt_Tuc).sum(axis=0)  # shape (K,)
    E = (S @ S) / (m2 * np.sqrt(T_total))
    return float(E)


def build_degree_sampler(deg: np.ndarray, seed: int = 0):
    """
    Build a sampler for p(u)=k_u/sum k_u using CDF + searchsorted.
    """
    rng = np.random.default_rng(seed)
    m2 = float(deg.sum())
    if m2 <= 0:
        raise ValueError("Sum of degrees is zero.")
    p = deg / m2
    cdf = np.cumsum(p)
    cdf[-1] = 1.0

    def sample_one() -> int:
        r = rng.random()
        return int(np.searchsorted(cdf, r, side="right"))

    return sample_one, rng


def mc_estimate_E(
    T_uc: np.ndarray,
    deg: np.ndarray,
    T_total: float,
    M: int,
    seed: int,
) -> float:
    """
    Degree-sampling Monte Carlo estimator (unbiased):
      E_hat = (2m) * (1/M) * sum_i sum_C sqrt(T_{u_i,C} T_{v_i,C}/T)
    """
    if M <= 0:
        raise ValueError("M must be positive.")

    m2 = float(deg.sum())
    if m2 <= 0:
        return 0.0

    sample_one, rng = build_degree_sampler(deg, seed=seed)
    inv_sqrt_T = 1.0 / np.sqrt(T_total)

    acc = 0.0
    # vectorized-ish inner: compute sqrt(T_u * T_v) then sum over C
    for _ in range(M):
        u = sample_one()
        v = sample_one()
        acc += float(np.sqrt(np.maximum(T_uc[u], 0.0) * np.maximum(T_uc[v], 0.0)).sum()) * inv_sqrt_T

    E_hat = m2 * acc / M
    return float(E_hat)


def parse_M_list(s: str) -> list[int]:
    # supports "200,500,1000" or "200 500 1000"
    parts = [p.strip() for p in s.replace(" ", ",").split(",") if p.strip()]
    Ms = [int(x) for x in parts]
    if any(M <= 0 for M in Ms):
        raise ValueError("All M must be positive integers.")
    return Ms

In [16]:
df = pd.read_csv("data/link_stream.csv")

In [17]:
K = 5
T_uc, deg, T_total = compute_T_uc(df, K=K, seed=42)

In [18]:
# Step 3: exact E
E0 = exact_E(T_uc, deg, T_total)
print(f"[Exact] E_exact = {E0:.10g}")
print(f"[Info] #nodes={len(deg)}, K={K}, T_total={T_total:.6g}, 2m={deg.sum():.6g}")


[Exact] E_exact = 5137951187
[Info] #nodes=986, K=5, T_total=6.94593e+07, 2m=664668


In [20]:
MS = [i for i in range(10000, 50000, 5000)]
trials = 30

bias_means = []
variances = []

for j, M in enumerate(MS):
    errs = np.empty(trials, dtype=np.float64)

    for t in range(trials):
        trial_seed = 42 * 10_000 + j * 1_000 + t
        E_hat = mc_estimate_E(T_uc, deg, T_total, M=M, seed=trial_seed)
        errs[t] = E_hat - E0   # <-- 误差

    bias = errs.mean()                 # mean(E_hat) - E0
    var = errs.std(ddof=1)

    bias_means.append(bias)
    variances.append(var)

    print(
        f"[MC] M={M:<6d}  "
        f"bias={bias:.4e}  "
        f"var={var:.4e}"
    )

# ---------- plotting ----------
labels = [str(M) for M in MS] + ["Exact"]
x = np.arange(len(labels))

y_vals = bias_means + [0.0]        # Exact 的 bias = 0
y_errs = variances + [0.0]          # Exact 没有方差

plt.figure(figsize=(8, 4))

plt.bar(
    x,
    y_vals,
    yerr=y_errs,
    capsize=6,
    alpha=0.7,
    edgecolor="black"
)

plt.axhline(0.0, linestyle="--", linewidth=1)

plt.xticks(x, labels)
plt.xlabel("Monte Carlo sample size")
plt.ylabel("mean(E_hat) − E_exact")
plt.title(f"Bias of null expectation estimator (K={K}, trials={trials})")

plt.tight_layout()
plt.savefig("results/bias_vs_sampling.png", dpi=160)
plt.close()

[MC] M=10000   bias=4.0120e+05  var=5.5802e+06
[MC] M=15000   bias=2.9736e+05  var=5.8721e+06
[MC] M=20000   bias=-1.2546e+06  var=5.0523e+06
[MC] M=25000   bias=-9.5609e+05  var=4.0935e+06
[MC] M=30000   bias=-9.3807e+04  var=4.1902e+06
[MC] M=35000   bias=4.0093e+04  var=3.2446e+06
[MC] M=40000   bias=-6.0919e+05  var=2.9342e+06
[MC] M=45000   bias=1.1662e+06  var=2.4940e+06
